In [0]:
# This question asked in Startup-> Celeritiasai 

oracle db to azure blob storage using Microsoft Fabric. Now notebook is given to you :
2 tables 1) Employee_Master contains following columns: EMP_ID, emp_name, emp_age, emp_doj, emp_dept
2nd table 2) Employee_Salary contains: emp_id, emp_name, emp_basic_pay, emp_total_pay

Problem Statement: from oracle db you need to pull out the data using microsoft fabric or rather PySpark to be very specific,you have to write etl job. so to pull out the data and need to final transform the data and the transformations need to happen in sql. Transformation should be like this the employees who are having salary of greater 5 lakhs and and employee age greater than 50 years are pushed into operations department.Remaining all should be there with finance department. There are different parts to it. Okay. There is a data engineering part to it.
As you defined the gold layer, silver layer, and bronze layer, you have all the three different distinctions here.
You tell me how would you take this entire project end to end or how would you define the entire pipeline end to end?
So, since you are getting interviewed for ETL, your main focus should be on the ETL part, on the PySpark part
So, I want you to load the PySpark jobs, open the PySpark sessions, create the sessions, and do all of that.

SELECT 
    m.EMP_ID, 
    m.emp_name, 
    m.emp_age, 
    m.emp_doj, 
    s.emp_basic_pay, 
    s.emp_total_pay,
    CASE 
        WHEN s.emp_total_pay > 500000 AND m.emp_age > 50 THEN 'Operations'
        ELSE 'Finance' 
    END AS emp_dept
FROM Employee_Master m
JOIN Employee_Salary s ON m.EMP_ID = s.emp_id;

🔷 1. End-to-End Pipeline Design (Interview Explanation)
✅ Data Source
Oracle DB (Employee tables)
✅ Target
Azure Blob Storage (via Microsoft Fabric Lakehouse)
✅ Architecture (Medallion)
Bronze → Raw ingestion (no transformation)
Silver → Cleaned + joined data
Gold → Business logic applied (department assignment)
🔷 2. High-Level Flow
Connect to Oracle DB using JDBC
Ingest both tables into Bronze Layer
Clean + join → move to Silver Layer
Apply business transformation using SQL → move to Gold Layer
Store final data in Azure Blob (Lakehouse tables/files)
🔷 3. PySpark Notebook Implementation
✅ Step 1: Start Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Oracle_to_Fabric_ETL") \
    .getOrCreate()
✅ Step 2: Oracle JDBC Connection
jdbc_url = "jdbc:oracle:thin:@//hostname:1521/servicename"

connection_properties = {
    "user": "your_username",
    "password": "your_password",
    "driver": "oracle.jdbc.driver.OracleDriver"
}
✅ Step 3: Load Tables (Bronze Layer)
emp_master_df = spark.read.jdbc(
    url=jdbc_url,
    table="Employee_Master",
    properties=connection_properties
)

emp_salary_df = spark.read.jdbc(
    url=jdbc_url,
    table="Employee_Salary",
    properties=connection_properties
)
Write to Bronze
emp_master_df.write.mode("overwrite").saveAsTable("bronze_employee_master")
emp_salary_df.write.mode("overwrite").saveAsTable("bronze_employee_salary")
🔷 4. Silver Layer (Cleaning + Join)
Basic Cleaning (optional but good to mention)
from pyspark.sql.functions import col

emp_master_clean = emp_master_df.dropDuplicates(["EMP_ID"])
emp_salary_clean = emp_salary_df.dropDuplicates(["EMP_ID"])
Join Tables
emp_joined_df = emp_master_clean.join(
    emp_salary_clean,
    emp_master_clean.EMP_ID == emp_salary_clean.emp_id,
    "inner"
).select(
    emp_master_clean.EMP_ID,
    "emp_name",
    "emp_age",
    "emp_doj",
    "emp_dept",
    "emp_basic_pay",
    "emp_total_pay"
)
Save Silver Table
emp_joined_df.write.mode("overwrite").saveAsTable("silver_employee")
🔷 5. Gold Layer (SQL Transformation)

👉 IMPORTANT: Interviewer clearly wants SQL transformation, not PySpark logic.

Create Temp View
emp_joined_df.createOrReplaceTempView("employee_silver")
✅ Business Logic
Salary > 5 Lakhs
Age > 50 → Operations
Else → Finance
✅ SQL Transformation
SELECT 
    EMP_ID,
    emp_name,
    emp_age,
    emp_doj,
    emp_basic_pay,
    emp_total_pay,
    CASE 
        WHEN emp_total_pay > 500000 AND emp_age > 50 THEN 'Operations'
        ELSE 'Finance'
    END AS final_department
FROM employee_silver
Execute SQL in PySpark
gold_df = spark.sql("""
SELECT 
    EMP_ID,
    emp_name,
    emp_age,
    emp_doj,
    emp_basic_pay,
    emp_total_pay,
    CASE 
        WHEN emp_total_pay > 500000 AND emp_age > 50 THEN 'Operations'
        ELSE 'Finance'
    END AS final_department
FROM employee_silver
""")
Save Gold Layer
gold_df.write.mode("overwrite").saveAsTable("gold_employee")
🔷 6. Load to Azure Blob Storage

In Microsoft Fabric, Lakehouse is already backed by OneLake (similar to ADLS).

gold_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .save("abfss://container@storageaccount.dfs.core.windows.net/gold/employee/")
🔷 7. Pipeline Orchestration (Very Important for Interview)

Explain this clearly 👇

Use Microsoft Fabric Data Pipeline:
Notebook Activity
Runs PySpark ETL
Schedule
Daily / Incremental
Error Handling
Retry logic
Logging
Monitoring
Pipeline run history
🔷 8. Incremental Load (Bonus Point 💯)

Instead of full load:

Use emp_doj / last_updated_date
Load only new records

Example:

WHERE emp_doj > last_run_timestamp
🔷 9. Key Interview Points to Say
Used JDBC for Oracle extraction
Followed Medallion Architecture
Used SQL for business transformation (as required)
Stored data in Lakehouse (Azure Blob)
Designed pipeline with scalability & reusability
Can extend with:
Partitioning
Delta tables
Slowly Changing Dimensions
🔷 Final One-Line Summary (Say This Confidently)

“I designed an end-to-end ETL pipeline in Microsoft Fabric using PySpark, where I ingested Oracle data into Bronze, performed joins and cleansing in Silver, and applied business transformations using SQL in Gold layer, finally storing the processed data in Azure Blob Storage and orchestrating via Fabric pipelines.”